In [5]:
import os
import random
import pandas as pd
from training import *
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=UserWarning)

# Setup and Dataloading
[A] - specify training sites, test sites and non-predictive columns <br>
> *data_dir* is used to create a site list, NOTE: *data_dir* therefor needs to contain one subdir per site <cr> <br>
    
[B] - load csv file that contains all training data (csv created by notebooks 1 \& 2) <br>
> possibility to filter by class certainty

In [2]:
#[A]
data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/2_labelling/hp'
sites = os.listdir(data_dir)
test_sites = ['CG1-8B', 'F3-20B', 'CS-103A', 'ZF20-11A'] # Don't train on these
#train_sites = [s for s in sites if s not in test_sites]
#random.shuffle(train_sites)
non_predictive_columns = ['x_pos', 'y_pos', 'site', 'class_certainty', 'veg_class']

In [3]:
#[B]

#original (untargetted) data
results_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/'
df = pd.read_csv(results_dir+'dataset_file_transformed.csv')
df_reduced = df[df['class_certainty'] <= 2]
non_predictive_columns = ['x_pos', 'y_pos', 'site', 'class_certainty', 'veg_class']

# Data with targetted labels added. Has extreme class imbalance
#df_targetted = pd.read_csv('dataset_targetted_transformed.csv')
#df_targetted_reduced = df_targetted[df_targetted['class_certainty'] <= 2]

# Hyperparameter Search
We use a custom cross validataion (CV) and random search so that we can split the train and validation data by site. These functions are in training.py

# Experiments
Since the data has high spatial (and site-level) autocorrelation, we have a custom procedure for running experiments.
In the above section, we defined a site-level cross-validation routine, and nested that inside a custom hyperparameter search
that uses that CV. So for each experiment, we just need to implement the core training logic.

- define a function first
  - inputs:
    - model - initialized sklearn model
    - train_data - dataframe with all data for train sites
    - val_data - dataframe with all data for validation sites
  - outputs:
    - train accuracy
    - val accuracy
  - this function will be used within the CV loop, nested inside random search

- define the data you will use (+/- targetted labels for example)
- define model type and parameter distribution

This structure allows for maximum flexibility with how you define an experiment. When prototyping, it is easiest to not use the
CV functionality, and instead structure it like this:

```
def experiment_fn(model, train_data, val_data):
    ...
    ...
    return model.score(X_res, y_res), model.score(X_train, y_train)

model = RandomForestClassifier({params})

experiment_fn(model, df[df['site'].isin(train_sites)], df[df['site'].isin(test_sites)])

```

Once the training function is working as expected, try plugging it into the random search routine.

## Baseline
- Original data (without targetted lichen labels)
- Random Forest with moderate hyperparameter search
- SMOTE oversampling for underrepresented classes

In [7]:
def experiment_fn(model, train_data, val_data, non_predictive_columns):
    X_train, y_train = split_xy(train_data, non_predictive_columns)
    X_val, y_val = split_xy(val_data, non_predictive_columns)
    
    # -------------------------------
    # oversample w synthetic minority oversampling to balance classes
    over = imblearn.over_sampling.SMOTE(sampling_strategy='not majority') # resamples all classes but majority
    X_res, y_res = over.fit_resample(X_train, y_train)

    # -----------------------------------------------------------------------------------------------------------
    # Train RF classifier
    model.fit(X_res.values, y_res.values)

    # -----------------------------------------------------------------------------------------------------------
    return model.score(X_res, y_res), model.score(X_val, y_val)


data = df_reduced

model_type = RandomForestClassifier    

param_dist = { # discrete search space for random search
    'n_estimators': [400, 500, 600],
    'max_depth': [6, 8, 10, 12, 14],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
}

# Run hyperparameter search with CV
random_search(data, model_type, param_dist, experiment_fn, n_iterations=10, test_sites=test_sites, non_predictive_columns=non_predictive_columns)

'''
model = model_type(n_estimators=500, max_depth=10, n_jobs=8)
experiment_fn(model, data[data['site'].isin(train_sites)], data[data['site'].isin(test_sites)])
'''

Trying {'n_estimators': 600, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 2}
  Val accuracies: [0.7042082954889495, 0.6023796347537355, 0.5953615279672578, 0.6626373626373626, 0.6716549295774648, 0.6368309646205019, 0.6536277602523659, 0.48186946011281223]
  Train accuracy mean: 0.879308546850121, Val accuracy mean: 0.6260712419263064
Trying {'n_estimators': 500, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 1}
  Val accuracies: [0.6156808803301238, 0.4896551724137931, 0.6754989676531314, 0.6343238326258338, 0.6813590449954087, 0.6492644937986732, 0.5776289682539683, 0.6961111111111111]
  Train accuracy mean: 0.826106632009453, Val accuracy mean: 0.6274403088977555
Trying {'n_estimators': 600, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 2}
  Val accuracies: [0.6411802030456852, 0.5947253810791193, 0.6314553990610329, 0.6628088833281202, 0.48413111342351717, 0.7074774034511093, 0.6417478283758884, 0.6985677083333334]
  Train accuracy mean: 

"\nmodel = model_type(n_estimators=500, max_depth=10, n_jobs=8)\nexperiment_fn(model, data[data['site'].isin(train_sites)], data[data['site'].isin(test_sites)])\n"

## Weighted classes
This one seems to be going quite well.

In [ ]:
def experiment_fn(model, train_data, val_data):
    X_train, y_train = split_xy(train_data)
    X_val, y_val = split_xy(val_data)
    X_res, y_res = X_train, y_train

    model.fit(X_res.values, y_res.values)

    return model.score(X_res, y_res), model.score(X_val, y_val)


data = df_targetted_reduced

model_type = RandomForestClassifier    

param_dist = { # discrete search space for random search
    'n_estimators': [400, 500, 600],
    'max_depth': [18, 20, 22, 24, 26, 28],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced', 'balanced_subsample'],
}

# Run hyperparameter search with CV
random_search(data, model_type, param_dist, experiment_fn, n_iterations=25)

'''
model = model_type(n_estimators=500, max_depth=10, n_jobs=8)
experiment_fn(model, data[data['site'].isin(train_sites)], data[data['site'].isin(test_sites)])
'''

## Naive targetted undersampling
- Targetted data included
- Random Forest with moderate hyperparameter search
- RandomUndersampler to balance lichen to other classes

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

def experiment_fn(model, train_data, val_data):
    X_train, y_train = split_xy(train_data)
    X_val, y_val = split_xy(val_data)
    
    under = imblearn.under_sampling.RandomUnderSampler(sampling_strategy='majority')
    X_res, y_res = under.fit_resample(X_train, y_train)

    model.fit(X_res.values, y_res.values)

    return model.score(X_res, y_res), model.score(X_val, y_val)




data = df_targetted_reduced

model_type = RandomForestClassifier    

param_dist = { # discrete search space for random search
    'n_estimators': [400, 500, 600],
    'max_depth': [6, 8, 10, 12, 14],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
}

# Run hyperparameter search with CV
random_search(data, model_type, param_dist, experiment_fn, n_iterations=5)

'''
model = model_type(n_estimators=500, max_depth=10, n_jobs=8)
experiment_fn(model, data[data['site'].isin(train_sites)], data[data['site'].isin(test_sites)])
'''

# Scratch Area
Playing around with the data, and informal experiments

In [ ]:
# Used for single experiments
def run_experiment(X_train, X_test, y_train, y_test, extra_metrics=False):
    # remove irrelevant columns
    X_train = X_train.drop(columns=non_predictive_columns)
    X_test = X_test.drop(columns=non_predictive_columns)

    # -------------------------------
    # oversample w synthetic minority oversampling to balance classes
    under = imblearn.over_sampling.SMOTE(sampling_strategy='not majority') # resamples all classes but majority
    X_res, y_res = under.fit_resample(X_train, y_train)

    # -----------------------------------------------------------------------------------------------------------
    # Train RF classifier
    clf=RandomForestClassifier(n_estimators=600, max_depth=6, n_jobs=12)
    clf.fit(X_res.values, y_res.values)

    # -----------------------------------------------------------------------------------------------------------
    # Prediction
    preds = clf.predict(X_test.values)

    print(f'  Train accuracy: {clf.score(X_res, y_res)}')
    print(f'  Test accuracy: {clf.score(X_test, y_test)}')

    if extra_metrics:
        # -----------------------------------------------------------------------------------------------------------
        # Inspect performance parameters: training and testing scores, confusion matrix, and feature importanceax1 = plt.figure(figsize=(5,5)).add_subplot(111)

        cf_matrix = confusion_matrix(y_test, preds,  labels = clf.classes_)
        disp = ConfusionMatrixDisplay(confusion_matrix=cf_matrix, display_labels=clf.classes_)
        disp.plot()
        plt.title('Confusion Matrix')

        feature_importance = pd.DataFrame(clf.feature_importances_, index=X_res.columns).sort_values(by=0, ascending=False)
        print(feature_importance)

    return clf, preds


X_train, X_test, y_train, y_test = split_by_sites(df_reduced, test_sites)
model, _ = run_experiment(X_train, X_test, y_train, y_test, extra_metrics=True)

In [ ]:
df_majority = df[(df['veg_class'] == 1) & (df['class_certainty'] == 1)]
df_minority = df[(df['veg_class'] != 1) & (df['class_certainty'] <= 3)]

# Get count of minority class
minority_count = len(df_minority)

# Randomly sample from majority class to match the count of minority class
df_majority_downsampled = df_majority.sample(n=minority_count//3, random_state=42)

# Combine minority class with downsampled majority class
df_downsampled = pd.concat([df_majority_downsampled, df_minority])

# Shuffle the dataset
df_reduced = df_downsampled.sample(frac=1, random_state=42).reset_index(drop=True)
df_reduced['veg_class'].value_counts()